## Heart Attack Prediction

In [12]:
# load the libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from joblib import dump, load
from sklearn.compose import ColumnTransformer

In [13]:
# load the data
# Assicurati di aver scaricato il CSV da Kaggle e messo in '../data/heart.csv'
data = pd.read_csv('../data/heart.csv')

In [14]:
# check the target variable distribution
# La colonna target in questo dataset si chiama 'output'
print(data.output.value_counts())

output
1    165
0    138
Name: count, dtype: int64


In [15]:
# check the first few rows of the data
data.head()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


### Data Preprocessing

In [17]:
### Data Preprocessing

# 1. Definiamo i tipi di colonne
# Colonne numeriche che vogliamo scalare
numerical_cols = ['age', 'trtbps', 'chol', 'thalachh', 'oldpeak']

# Colonne categoriche (già in formato numerico) che lasceremo invariate
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exng', 'slp', 'caa', 'thall']

# Colonna Target
target_col = 'output'

# 2. Separiamo features (x) e target (y)
y = data[target_col]
x = data.drop(target_col, axis = 1)

# 3. Creiamo il ColumnTransformer
# Questo oggetto applicherà lo StandardScaler SOLO alle 'numerical_cols'
# e lascerà invariate ('passthrough') le 'categorical_cols'
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', 'passthrough', categorical_cols)
    ],
    remainder='drop' # Ignora colonne non specificate
)

# 4. Applichiamo il fit_transform
# 'features' ora è il nostro set di dati processato
features_processed = preprocessor.fit_transform(x)

# 5. Ricreiamo il DataFrame processato (per coerenza)
# L'ordine delle colonne sarà prima 'numerical_cols' e poi 'categorical_cols'
final_columns_ordered = numerical_cols + categorical_cols
features = pd.DataFrame(features_processed, columns = final_columns_ordered)

# Stampa di controllo
print(f"Ordine finale delle colonne: {final_columns_ordered}")
features.head()

Ordine finale delle colonne: ['age', 'trtbps', 'chol', 'thalachh', 'oldpeak', 'sex', 'cp', 'fbs', 'restecg', 'exng', 'slp', 'caa', 'thall']


,age,trtbps,chol,thalachh,oldpeak,sex,cp,fbs,restecg,exng,slp,caa,thall
0,0.952197,0.763956,-0.256334,0.015443,1.087338,1.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0
1,-1.915313,-0.092738,0.072199,1.633471,2.122573,1.0,2.0,0.0,1.0,0.0,0.0,0.0,2.0
2,-1.474158,-0.092738,-0.816773,0.977514,0.310912,0.0,1.0,0.0,0.0,0.0,2.0,0.0,2.0
3,0.180175,-0.663867,-0.198357,1.239897,-0.206705,1.0,1.0,0.0,1.0,0.0,2.0,0.0,2.0
4,0.290464,-0.663867,2.082050,0.583939,-0.379244,0.0,0.0,0.0,1.0,1.0,2.0,0.0,2.0


In [18]:
# save the preprocessor (che agisce come il nostro scaler)
dump(preprocessor, '../models/scaler.joblib')

['../models/scaler.joblib']

### Modelling

In [19]:
# split the data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(features, y, test_size = 0.2, random_state = 42)

In [20]:
# define the model
model = MLPClassifier(solver="adam", max_iter=1000, activation="relu", hidden_layer_sizes=(12), alpha=0.001, batch_size=32, learning_rate_init=0.0001, random_state=42, verbose=True)
# fit the model
model.fit(x_train, y_train.values.ravel())

Iteration 1, loss = 0.71763246
Iteration 2, loss = 0.71578716
Iteration 3, loss = 0.71404768
Iteration 4, loss = 0.71235432
Iteration 5, loss = 0.71074342
Iteration 6, loss = 0.70900422
Iteration 7, loss = 0.70740038
Iteration 8, loss = 0.70568009
Iteration 9, loss = 0.70409167
Iteration 10, loss = 0.70252172
Iteration 11, loss = 0.70081863
Iteration 12, loss = 0.69930666
Iteration 13, loss = 0.69766307
Iteration 14, loss = 0.69615634
Iteration 15, loss = 0.69460629
Iteration 16, loss = 0.69300566
Iteration 17, loss = 0.69156178
Iteration 18, loss = 0.69003570
Iteration 19, loss = 0.68856291
Iteration 20, loss = 0.68701144
Iteration 21, loss = 0.68563563
Iteration 22, loss = 0.68413174
Iteration 23, loss = 0.68265190
Iteration 24, loss = 0.68121077
Iteration 25, loss = 0.67986676
Iteration 26, loss = 0.67837770
Iteration 27, loss = 0.67694687
Iteration 28, loss = 0.67554334
Iteration 29, loss = 0.67424915
Iteration 30, loss = 0.67276592
Iteration 31, loss = 0.67139889
Iteration 32, los

MLPClassifier(alpha=0.001, batch_size=32, hidden_layer_sizes=12,
              learning_rate_init=0.0001, max_iter=1000, random_state=42,
              verbose=True)

In [21]:
# evaluate the model
y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87        29
           1       0.90      0.84      0.87        32

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.87      0.87      0.87        61



In [22]:
# save the trained model
dump(model, '../models/model.joblib')

['../models/model.joblib']

### Inference

In [23]:
# Usiamo 'x' (dati pre-processamento) per prendere una riga media come test
test_row_values = x.mean()
# Creiamo un DataFrame con questa riga
test_df = pd.DataFrame([test_row_values], columns=x.columns) # IMPORTANTE: deve essere un DataFrame

# load the scaler (preprocessor) and model
preprocessor = load('../models/preprocessor_heart.joblib')
model = load('../models/model_heart.joblib')

# transform the new data
# Il preprocessor si aspetta il DataFrame
features_test_processed = preprocessor.transform(test_df)

# make a prediction
if (model.predict(features_test_processed)==0):
    print("Predizione: Basse probabilità di infarto.")
else:
    print("Predizione: Alte probabilità di infarto.")

Predizione: Basse probabilità di infarto.


C:\Users\sasyc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
